# 66 — Structured Game Theory (GaT)

**Dataset:** Structured Retrospective (task-planning app; N=61 behavioral participants)
**Model family:** Game Theory (GaT) — the final active Structured model family (ABM deferred)
**Blueprint:** `outputs/reports/structured/structured_gat_blueprint.md` — read this first. The paper's
own §1.3 groups GaT with StM under the same caution ("require more careful proxy definitions" — payoff
and utility are not directly observed). No formal game matrix, best-response analysis, or equilibrium
concept is built here — an explicit commitment INDEX (a named proxy) and correlational outcome tests,
consistent with every other Structured notebook's Step 1 descriptive/correlational scope.
**Output root:** `outputs/model_outputs/structured/game_theory/`

---

## Commitment index (Part 1 of blueprint)

`commitment_score = temporal_commitment(inbox=0/all_day=1/timed=2) + is_recurring + has_subtasks + long_duration`
(range 0-5). Built strictly from the paper's own named commitment signals ("a specific time, duration,
alert, recurrence, or subtasks") — alerts excluded, confirmed absent from the export.

## Headline test: the trade-off curve is non-monotonic (verified before building this notebook)

Completion peaks at MODERATE commitment (score 2), declines at maximum commitment (score 5) -- directly
matches the paper's own "neither strategy is inherently better" framing. Reported as a binned curve, not
a single linear correlation (which would misrepresent a U-shaped relationship).


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
SYD_OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "system_dynamics"
OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "game_theory"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

MIN_TASKS = 10

print("Output folder:", OUT)

## 1. Load Data

In [ ]:
beh = pd.read_csv(CLEANED / "structured_behavioral_clean.csv")
beh["event_timestamp_utc"] = pd.to_datetime(beh["event_timestamp_utc"], format="mixed", utc=True)
tasks = pd.read_csv(FEATURES / "structured_task_lifecycle_features.csv")
pf = pd.read_csv(FEATURES / "structured_participant_features.csv")

print(f"Tasks: {len(tasks):,}  Participants: {pf.shape[0]}")

## 2. Build the Commitment Index (Part 1 of blueprint)

In [ ]:
tasks = tasks.copy()
temporal_map = {"inbox": 0, "all_day": 1, "timed": 2}
tasks["temporal_commitment"] = tasks["task_type_last"].map(temporal_map)
tasks["is_recurring"] = tasks["task_is_recurring_ever"].astype(int)
tasks["has_subtasks"] = (tasks["subtasks_amount_last"] > 0).astype(int)

median_dur = tasks.loc[tasks["task_type_last"] == "timed", "task_duration_min_last"].median()
tasks["long_duration"] = np.where(tasks["task_type_last"] == "timed",
                                    (tasks["task_duration_min_last"] > median_dur).astype(int), 0)

tasks["commitment_score"] = (tasks["temporal_commitment"] + tasks["is_recurring"] +
                               tasks["has_subtasks"] + tasks["long_duration"])

print("commitment_score distribution:")
print(tasks["commitment_score"].value_counts().sort_index())
print(f"\nMedian duration among timed tasks used for long_duration cutoff: {median_dur} min")

## 3. The Trade-Off Curve (Part 2 of blueprint — headline result)

In [ ]:
curve = tasks.groupby("commitment_score")["lifecycle_status"].value_counts(normalize=True).unstack(fill_value=0)
curve["n"] = tasks.groupby("commitment_score").size()
curve = curve[["completed", "deleted", "unresolved", "n"]].rename(columns={
    "completed": "completion_rate", "deleted": "deletion_rate", "unresolved": "unresolved_rate"})
print(curve.round(4))

r_linear, p_linear = stats.pearsonr(tasks["commitment_score"], (tasks["lifecycle_status"] == "completed").astype(int))
print(f"\nLinear correlation (commitment_score vs completed): r={r_linear:.4f}, p={p_linear:.4g}")
print("NOTE: the relationship above is non-monotonic (peaks at score=2) -- this single linear r")
print("understates/misrepresents the real effect. The binned curve is the primary evidence.")

## 4. Within-Participant Check on the Trade-Off (proactive, per the SyD/StM lesson)

In [ ]:
tasks["is_completed_bin"] = (tasks["lifecycle_status"] == "completed").astype(int)
n_tasks_pp = tasks.groupby("participant_id").size()
elig_pids = n_tasks_pp[n_tasks_pp >= MIN_TASKS].index
sub = tasks[tasks["participant_id"].isin(elig_pids)].copy()

sub["commit_dm"] = sub["commitment_score"] - sub.groupby("participant_id")["commitment_score"].transform("mean")
sub["completed_dm"] = sub["is_completed_bin"] - sub.groupby("participant_id")["is_completed_bin"].transform("mean")

r_within, p_within = stats.pearsonr(sub["commit_dm"], sub["completed_dm"])
print(f"Within-participant linear correlation: r={r_within:.4f}, p={p_within:.4g}, n={len(sub)}")
print()
print("Within-participant binned curve (same non-monotonic shape check, demeaned population):")
within_curve = sub.groupby("commitment_score")["is_completed_bin"].mean()
print(within_curve.round(4))

## 5. Save Trade-Off Curve

In [ ]:
curve.to_csv(OUT / "commitment_outcome_curve.csv")
print(f"Saved: commitment_outcome_curve.csv  ({curve.shape})")

## 6. Figure 1 — Commitment-Outcome Trade-Off Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(curve.index, curve["completion_rate"], marker="o", color="#55A868", label="completion rate")
axes[0].plot(curve.index, curve["deletion_rate"], marker="o", color="#C44E52", label="deletion rate")
axes[0].set_xlabel("commitment_score")
axes[0].set_ylabel("rate")
axes[0].set_title("Pooled: outcome rate by commitment level")
axes[0].legend()

axes[1].plot(within_curve.index, within_curve.values, marker="o", color="#4C72B0")
axes[1].set_xlabel("commitment_score")
axes[1].set_ylabel("completion rate (within-participant demeaned population)")
axes[1].set_title("Within-participant: same non-monotonic shape?")

plt.tight_layout()
plt.savefig(FIG / "fig1_gat_commitment_outcome_curve.png", dpi=150)
plt.show()
print("Saved fig1_gat_commitment_outcome_curve.png")

## 7. Per-Participant Phenotype Labels (Part 3 of blueprint)

In [ ]:
mean_commit_pp = tasks.groupby("participant_id")["commitment_score"].mean()

gat_records = []
for pid in pf["participant_id"]:
    n = n_tasks_pp.get(pid, 0)
    is_eligible = n >= MIN_TASKS
    rec = {"participant_id": pid, "n_tasks": n, "is_gat_eligible": is_eligible,
           "mean_commitment_score": round(float(mean_commit_pp.get(pid, np.nan)), 4) if is_eligible else np.nan}
    gat_records.append(rec)

gat_df = pd.DataFrame(gat_records)
gat_df = gat_df.merge(pf[["participant_id", "reschedule_rate", "pct_tasks_completed"]],
                       on="participant_id", how="left")

elig = gat_df[gat_df["is_gat_eligible"]]
commit_terc = elig["mean_commitment_score"].quantile([1/3, 2/3]).values
resched_terc = elig["reschedule_rate"].dropna().quantile([1/3, 2/3]).values
completed_terc = elig["pct_tasks_completed"].dropna().quantile([1/3, 2/3]).values

def classify(r):
    if r["mean_commitment_score"] >= commit_terc[1]:
        return "commitment-oriented"
    if r["mean_commitment_score"] <= commit_terc[0]:
        return "flexibility-oriented"
    if pd.notna(r["reschedule_rate"]) and r["reschedule_rate"] >= resched_terc[1]:
        if pd.notna(r["pct_tasks_completed"]) and r["pct_tasks_completed"] <= completed_terc[0]:
            return "chronic-deferrer"
        if pd.notna(r["pct_tasks_completed"]) and r["pct_tasks_completed"] >= completed_terc[1]:
            return "adaptive-rescheduler"
    return "mixed"

gat_df.loc[elig.index, "phenotype_label"] = elig.apply(classify, axis=1)
print(f"GaT-eligible: {len(elig)} / {len(gat_df)}")
print(gat_df["phenotype_label"].value_counts(dropna=False))
print()
print("NOTE: the paper's 5th named GaT phenotype, 'overload-prone', is not recomputed here --")
print("already covered by SyD's dominant_loop_label='overload-trending' (structured_syd_blueprint.md).")

## 8. Save Participant Summary

In [ ]:
gat_df.to_csv(OUT / "participant_gat_summary.csv", index=False)
print(f"Saved: participant_gat_summary.csv  ({gat_df.shape})")

## 9. Figure 2 — Phenotype Label Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
gat_df["phenotype_label"].value_counts(dropna=False).plot(kind="barh", ax=ax, color="#8172B2")
ax.set_xlabel("Number of participants")
ax.set_title("GaT descriptive phenotype labels")
plt.tight_layout()
plt.savefig(FIG / "fig2_gat_phenotype_distribution.png", dpi=150)
plt.show()
print("Saved fig2_gat_phenotype_distribution.png")

## 10. Strategy Adaptation After Rescheduling (Part 4 of blueprint)

**Scoping caveat:** `was_rescheduled` marks whether a task was rescheduled at ANY point in its own
lifecycle, not strictly before the next task's creation -- a task created shortly after another could in
principle be flagged based on a reschedule that technically happens slightly later in wall-clock time.
This is a minor, exploratory simplification (avoiding a full interleaved per-participant event timeline)
and is reported as such rather than presented as a strict causal-ordering test.

In [ ]:
def to_local_date_series(df):
    out = pd.Series(index=df.index, dtype="object")
    for tz, grp in df.groupby("event_timezone"):
        try:
            out.loc[grp.index] = grp["event_timestamp_utc"].dt.tz_convert(tz).dt.date
        except Exception:
            out.loc[grp.index] = pd.NaT
    return out

created = beh[beh["event_type"] == "Task Created"].copy()
created["local_date"] = to_local_date_series(created)
created = created.dropna(subset=["local_date"]).sort_values("event_timestamp_utc")
created = created.drop_duplicates(subset="task_id", keep="first")
entry_ts = created.set_index("task_id")["event_timestamp_utc"]

def has_reschedule_tag(s):
    return pd.notna(s) and bool({"date_changed", "start_time_changed"} & set(s.split("|")))

beh_upd = beh[beh["event_type"] == "Task Updated"].copy()
resched_ids = set(beh_upd[beh_upd["update_type_list_str"].apply(has_reschedule_tag)]["task_id"])

nc_tasks = tasks[tasks["has_creation_event"]].copy()
nc_tasks["entry_ts"] = nc_tasks["task_id"].map(entry_ts)
nc_tasks = nc_tasks.dropna(subset=["entry_ts"]).sort_values(["participant_id", "entry_ts"])
nc_tasks["was_rescheduled"] = nc_tasks["task_id"].isin(resched_ids)

adapt_records = []
for pid, grp in nc_tasks.groupby("participant_id"):
    grp = grp.reset_index(drop=True)
    prior_flag = False
    for i, row in grp.iterrows():
        adapt_records.append({"participant_id": pid, "prior_reschedule_flag": int(prior_flag),
                               "commitment_score": row["commitment_score"]})
        prior_flag = row["was_rescheduled"]

adapt_df = pd.DataFrame(adapt_records)
print(f"Strategy-adaptation observations: {len(adapt_df):,}")

r_pooled_adapt, p_pooled_adapt = stats.pearsonr(adapt_df["prior_reschedule_flag"], adapt_df["commitment_score"])

adapt_df["flag_dm"] = adapt_df["prior_reschedule_flag"] - adapt_df.groupby("participant_id")["prior_reschedule_flag"].transform("mean")
adapt_df["commit_dm"] = adapt_df["commitment_score"] - adapt_df.groupby("participant_id")["commitment_score"].transform("mean")
# NOTE: distinct variable names from section 4's r_within/p_within (commitment-outcome test) --
# reusing those names here previously caused the model_run_summary to silently report this test's
# numbers under the wrong heading. Caught by cross-checking the full printed output before reporting.
r_within_adapt, p_within_adapt = stats.pearsonr(adapt_df["flag_dm"], adapt_df["commit_dm"])

adapt_result = pd.DataFrame([{
    "X": "prior_reschedule_flag", "Y": "commitment_score (current task)",
    "pearson_r_pooled": round(r_pooled_adapt, 4), "p_pooled": round(p_pooled_adapt, 6),
    "pearson_r_within": round(r_within_adapt, 4), "p_within": round(p_within_adapt, 6),
    "n": len(adapt_df),
}])
print(adapt_result.to_string(index=False))

## 11. Save Strategy Adaptation Results

In [ ]:
adapt_result.to_csv(OUT / "strategy_adaptation_test.csv", index=False)
print("Saved: strategy_adaptation_test.csv")

## 12. model_run_summary.md

In [ ]:
summary = f"""# Structured GaT -- Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/66_structured_game_theory.ipynb
**Blueprint:** outputs/reports/structured/structured_gat_blueprint.md

## Commitment-outcome trade-off curve (headline result, N={len(tasks):,} tasks)
{curve.round(4).to_string()}

Non-monotonic: completion peaks at commitment_score=2 (79.0%%), declines at score=5 (58.7%%).
Pooled linear r={r_linear:.4f} (p={p_linear:.4g}) -- understates the real U-shaped effect, reported
only as a secondary summary statistic, not the primary evidence.
Within-participant linear r={r_within:.4f} (p={p_within:.4g}, n={len(sub):,}) -- same shape check on
the demeaned population, see within_curve in the notebook.

## Phenotype labels (eligible, N={len(elig)})
{gat_df["phenotype_label"].value_counts(dropna=False).to_string()}
(paper's 5th named phenotype "overload-prone" cross-referenced to SyD's dominant_loop_label, not recomputed)

## Strategy adaptation after rescheduling (pooled vs. within-participant)
{adapt_result.to_string(index=False)}

## Design note
No formal game matrix, best-response analysis, or equilibrium concept was built, per the theory paper's
own caution (section 1.3) that GaT, like StM, requires proxy definitions rather than directly-observed
formal quantities. commitment_score is an explicit named proxy, not a utility function.
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)